# 13 pamoka – Agentų atmintis


## Diegimas

Šiame užrašų knygelėje demonstruojama, kaip sukurti kelionių rezervavimo agentą su **pastovia atmintimi** naudojant **Microsoft Agent Framework** (MAF).

Sužinosite, kaip skirtingi agento atminties tipai – darbo, trumpalaikė ir ilgalaikė – veikia tai, kaip agentas saugo ir naudoja informaciją per pokalbius.

**Reikalavimai:**
- Microsoft Foundry projektas su paleistu pokalbių modeliu (pvz., `gpt-4.1-mini`).
- Prisijungta prie Azure CLI – terminale paleiskite `az login`.
- `AZURE_AI_PROJECT_ENDPOINT` – jūsų Microsoft Foundry projekto galinis taškas.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` – jūsų paleisto modelio pavadinimas.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Agentų atminties tipai

DI agentai gali naudoti skirtingas atminties rūšis, kiekviena atliekanti skirtingą funkciją:

### Darbinė atmintis
Pačio pokalbio grandinė — žinutės, keistos per vieną sesiją. Agentas gali grįžti prie ankstesnių žinučių toje pačioje grandinėje, kad išlaikytų nuoseklumą. MAF tai sukuriama per **`agent.create_session()`**, kuris grąžina `AgentSession`.

### Trumpalaikė atmintis
Informacija, kuri išlieka tol, kol vyksta užduotis ar sesija, bet nėra saugoma nuolat. Pavyzdžiui, agentas gali sukaupti faktus daugiasluoksnio planavimo pokalbio metu ir panaudoti juos sukuriant galutinį maršrutą.

### Ilgalaikė atmintis
Nuostatos ir faktai, kurie išlieka **per sesijas**. Grįžtantis vartotojas neturėtų kartoti savo mitybos apribojimų ar kelionių stiliaus. Ilgalaikė atmintis dažniausiai palaikoma išorinėje saugykloje — duomenų bazėje, faile ar vektorių indekse — ir pateikiama agentui per įrankius.


## Darbinė atmintis su sesijomis

Paprasčiausia atminties forma yra pokalbio sesija. Kai perduodate tą pačią sesiją (sukurtą per `agent.create_session()`) paeiliui `agent.run()` kvietimams, agentas mato visą to pokalbio istoriją ir gali prisiminti ankstesnius duomenis.

Sukurkime kelionių agentą ir parodykime darbinę atmintį.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agentas teisingai prisiminė biudžetą, nes abu pranešimai priklauso tai pačiai sesijai. Tai yra **darbinė atmintis** — ji egzistuoja tik sesijos metu.

### Kas vyksta su nauju pokalbio giju?

Jei sukursime **naują** sesiją, agentas nebeturės atminties apie ankstesnį pokalbį:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Ilgalaikės atminties modelis

Norint prisiminti vartotojo nuostatas **per kelias sesijas**, mums reikia nuolatinės saugyklos, kuri būtų nepriklausoma nuo pokalbio gijos. Agentas prieina prie šios saugyklos per **įrankius** — funkcijas, kurias gali kviesti norėdamas išsaugoti ir gauti informaciją.

Žemiau įgyvendiname paprastą atminties nuostatų saugyklą (gyvenimo aplinkoje jūs naudotumėte duomenų bazę arba vektorinį indeksą) ir pateikiame ją kaip įrankius, kuriuos agentas gali naudoti.

### Architektūra
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenarijus 1 — Pirmą kartą vartotojas užsako jubiliejaus kelionę

Sarah lankosi pirmą kartą. Agentas turėtų įrašyti jos pageidavimus naudodamasis įrankiais ir juos naudoti viešbučiams rekomenduoti.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenarijus 2 — Sarah sugrįžta po kelių savaičių

Sarah pradeda **visiškai naują temą** (simuliuojant naują sesiją). Darbinė atmintis yra tuščia, tačiau ilgalaikėje pageidavimų saugykloje vis dar yra jos informacija. Agentas turėtų ją atkurti ir naudoti asmeninėms rekomendacijoms pateikti.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Santrauka

Šiame pamokoje sužinojote tris agento atminties tipus ir kaip juos įgyvendinti naudojant Microsoft Agent Framework:

| Atminties tipas | MAF mechanizmas | Gyvavimo trukmė |
|---|---|---|
| **Darbinė** | `agent.create_session()` | Viena pokalbio sesija |
| **Trumpalaikė** | Kauptas kontekstas pokalbio gijoje | Vienas uždavinys / sesija |
| **Ilgalaikė** | Išorinė saugykla, prieinama per `@tool` funkcijas | Tarp sesijų |

### Pagrindinės išvados
1. **`agent.create_session()`** suteikia darbinę atmintį — agentas mato visą pokalbio istoriją sesijos metu.
2. **Naujos sesijos praranda kontekstą** — be ilgalaikės atminties agentas negali prisiminti ankstesnių pokalbių.
3. **`@tool` funkcijos** užpildo spragą — jos leidžia agentui saugoti ir atkurti informaciją iš nuolatinės saugyklos.
4. **Personalizacija gerėja laikui bėgant** — kuo daugiau pageidavimų saugoma, tuo geresnės agento rekomendacijos.

### Realaus pasaulio taikymai
- **Klientų aptarnavimas**: prisiminti kliento istoriją ir pageidavimus
- **Asmeniniai padėjėjai**: palaikyti kontekstą per dienas ar savaites
- **Sveikatos priežiūra**: stebėti paciento informaciją ir pageidavimus
- **Elektroninė prekyba**: personalizuoti pirkimus pagal istoriją

### Tolimesni žingsniai
- Pakeisti atminties žodyną duomenų baze arba vektorių saugykla (pvz., Azure AI Search)
- Pridėti atminties galiojimo laiką jautriai informacijai
- Kurti daugiaagentines sistemas su bendromis atmintimis
- Išnagrinėti Cognee užrašų knygelę su žinių grafu paremta atmintimi


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
